In [272]:
import requests 
import pandas as pd 
import numpy as np
import sys 
import os
import geopandas as gpd
import json
from scipy.spatial import cKDTree
import importlib

sys.path.append(os.path.abspath("../Connectivity/model/"))
import dist_func
importlib.reload(dist_func)
from dist_func import nearest_neighbor, find_line_diversity, combine_connectivity_score, load_tfl

### Loading London underground, Elizabeth, overground, DLR station location data, and Grids

In [273]:
# underground
url1 = f"https://services1.arcgis.com/YswvgzOodUvqkoCN/ArcGIS/rest/services/TfL_stations/FeatureServer/1/query"
load_tfl(url1, "underground")

#Elizabeth
url2 = f"https://services1.arcgis.com/YswvgzOodUvqkoCN/ArcGIS/rest/services/TfL_stations/FeatureServer/12/query"
load_tfl(url2, "Elizabeth")

#DLR
url3 = f"https://services1.arcgis.com/YswvgzOodUvqkoCN/ArcGIS/rest/services/TfL_stations/FeatureServer/4/query"
load_tfl(url3, "DLR")


Total records: 273
Exported to data\underground.geojson
Total records: 41
Exported to data\Elizabeth.geojson
Total records: 45
Exported to data\DLR.geojson


### Add lat and lon to location data 

In [274]:
gdf_underground = gpd.read_file("data/underground.geojson")
gdf_elizabeth = gpd.read_file("data/Elizabeth.geojson")
gdf_dlr = gpd.read_file("data/DLR.geojson")
grids = gpd.read_file("data/grids.geojson")

# Source files contain British National Grid coordinates; fix CRS metadata if needed
gdf_underground = gdf_underground.set_crs(epsg=27700, allow_override=True)
gdf_elizabeth = gdf_elizabeth.set_crs(epsg=27700, allow_override=True)
gdf_dlr = gdf_dlr.set_crs(epsg=27700, allow_override=True)

# (Paddington appears twice in source)
gdf_underground = gdf_underground.drop_duplicates(subset=["NAME"], keep="first").reset_index(drop=True)

def add_lat_lon_columns(gdf_in):
    gdf_out = gdf_in.copy()
    gdf_wgs84 = gdf_out.to_crs(epsg=4326)
    gdf_out["lat"] = gdf_wgs84.geometry.y
    gdf_out["lon"] = gdf_wgs84.geometry.x
    return gdf_out

# Keep geometry in EPSG:27700, add lon/lat columns for reference and mapping
gdf_underground = add_lat_lon_columns(gdf_underground)
gdf_elizabeth = add_lat_lon_columns(gdf_elizabeth)
gdf_dlr = add_lat_lon_columns(gdf_dlr)

# Optional combined network dataset for downstream steps
gdf = pd.concat([gdf_underground, gdf_elizabeth, gdf_dlr], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=gdf_underground.crs)

print("Loaded stations:")
print("  Underground:", len(gdf_underground))
print("  Elizabeth:", len(gdf_elizabeth))
print("  DLR:", len(gdf_dlr))
print("  Combined:", len(gdf))

print("\nCRS check:")
print("  Underground CRS:", gdf_underground.crs)
print("  Elizabeth CRS:", gdf_elizabeth.crs)
print("  DLR CRS:", gdf_dlr.crs)

gdf_underground.head()

Loaded stations:
  Underground: 272
  Elizabeth: 41
  DLR: 45
  Combined: 358

CRS check:
  Underground CRS: EPSG:27700
  Elizabeth CRS: EPSG:27700
  DLR CRS: EPSG:27700


,OBJECTID,NAME,LINES,ATCOCODE,MODES,ACCESSIBILITY,NIGHT_TUBE,NETWORK,DATASET_LAST_UPDATED,FULL_NAME,geometry,lat,lon
0,111,St. Paul's,Central,940GZZLUSPU,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,St. Paul's station,POINT (532108.351 181274.188),51.514955,-0.097492
1,112,Mile End,"District, Hammersmith & City, Central",940GZZLUMED,"bus, tube",Partially Accessible - Interchange Only,Yes,London Underground,1638144000000,Mile End station,POINT (536500.783 182534.494),51.525236,-0.033741
2,113,Bethnal Green,Central,940GZZLUBLG,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,Bethnal Green station,POINT (535043.542 182718.411),51.527239,-0.054663
3,114,Leyton,Central,940GZZLULYN,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,Leyton station,POINT (538367.437 186075.854),51.556605,-0.005460
4,115,Snaresbrook,Central,940GZZLUSNB,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,Snaresbrook station,POINT (540156.771 188804.041),51.580678,0.021420


### All in one cell
1. Find k nearest stations using kd-tree
2. Find k nearest stations using kd-tree

### For sensitivity
1. k
2. alpha and beta

In [262]:
result = nearest_neighbor(grids, gdf, k=5)
result = find_line_diversity(result, gdf, k=5)
result = combine_connectivity_score(result, alpha=0.5, beta=0.5)

cols = [
    'harmonic_mean_adj_dist_norm',
    'line_unique_count_norm',
    'diversity_penalty',
    'connectivity_score'
]

print('Computed columns present:', [c for c in cols if c in result.columns])
print('connectivity_score range:', result['connectivity_score'].min(), 'to', result['connectivity_score'].max())

result.head()

Total 2464 grid points
Mapping 358 stations
Computed columns present: ['harmonic_mean_adj_dist_norm', 'line_unique_count_norm', 'connectivity_score']
connectivity_score range: 0.006838465292791769 to 0.9285714285714286


,lat,lon,nearest_station_1,nearest_station_1_dist,nearest_station_1_lat,nearest_station_1_long,nearest_station_2,nearest_station_2_dist,nearest_station_2_lat,nearest_station_2_long,...,nearest_station_4_long,nearest_station_5,nearest_station_5_dist,nearest_station_5_lat,nearest_station_5_long,harmonic_mean_adj_dist,harmonic_mean_adj_dist_norm,line_unique_count,line_unique_count_norm,connectivity_score
0,51.514517,-0.412511,Hayes & Harlington,1771.244817,51.503037,-0.419357,Southall,3416.473714,51.506146,-0.377104,...,-0.398846,Northolt,6282.635641,51.548260,-0.368629,3670.393986,0.099671,2,0.714286,0.406978
1,51.348731,-0.145405,Morden,8970.331885,51.402537,-0.194730,Colliers Wood,10464.843890,51.418159,-0.178033,...,-0.168365,Wimbledon,11877.704847,51.421434,-0.206491,10581.778161,0.295823,2,0.714286,0.505054
2,51.545036,-0.050901,Bethnal Green,2596.018455,51.527239,-0.054663,Mile End,3254.887245,51.525236,-0.033741,...,-0.025176,Whitechapel,3667.535970,51.520237,-0.059412,3231.858834,0.087224,3,0.571429,0.329327
3,51.527064,-0.051669,Bethnal Green,271.295125,51.527239,-0.054663,Stepney Green,881.913840,51.521863,-0.046564,...,-0.060742,Mile End,1638.690767,51.525236,-0.033741,715.194819,0.015799,3,0.571429,0.293614
4,51.459390,-0.328057,Richmond,2445.762292,51.463118,-0.301647,Hounslow East,3246.110081,51.473263,-0.356313,...,-0.366273,Kew Gardens,4646.712579,51.476998,-0.285019,3444.102271,0.093248,2,0.714286,0.403767


### Validation
1. City-wise score
2. Single grid point

In [263]:
import folium
import branca.colormap as cm

# Build map-ready points (EPSG:4326) from result dataframe
if isinstance(result, gpd.GeoDataFrame) and "geometry" in result.columns and result.crs is not None:
    result_map = result.to_crs(epsg=4326).copy()
elif {"x", "y"}.issubset(result.columns):
    result_map = gpd.GeoDataFrame(
        result.copy(),
        geometry=gpd.points_from_xy(result["x"], result["y"]),
        crs="EPSG:27700",
    ).to_crs(epsg=4326)
elif {"lon", "lat"}.issubset(result.columns):
    is_projected_like = (result["lon"].abs().max() > 180) or (result["lat"].abs().max() > 90)
    in_crs = "EPSG:27700" if is_projected_like else "EPSG:4326"
    result_map = gpd.GeoDataFrame(
        result.copy(),
        geometry=gpd.points_from_xy(result["lon"], result["lat"]),
        crs=in_crs,
    ).to_crs(epsg=4326)
else:
    raise ValueError("result must contain geometry, x/y, or lon/lat columns")

# Center map around all grid points
center_lat = float(result_map.geometry.y.mean())
center_lon = float(result_map.geometry.x.mean())
m_all = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles=None)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m_all)
folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m_all)

vmin = float(result_map["connectivity_score"].min())
vmax = float(result_map["connectivity_score"].max())
cmap = cm.LinearColormap(
    ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"],
    vmin=vmin,
    vmax=vmax,
 )
cmap.caption = "Connectivity Score"

for _, r in result_map.iterrows():
    folium.CircleMarker(
        location=[float(r.geometry.y), float(r.geometry.x)],
        radius=5.0,
        color=cmap(float(r["connectivity_score"])),
        fill=True,
        fill_opacity=0.8,
        weight=0,
        popup=(
            f"score={r['connectivity_score']:.3f}<br>"
            f"line_unique={r.get('line_unique_count', float('nan'))}<br>"
            f"nearest={r.get('nearest_station_1', 'NA')}"
        ),
    ).add_to(m_all)

cmap.add_to(m_all)
folium.LayerControl(collapsed=False).add_to(m_all)

city_html = os.path.abspath("connectivity_citywise_score_map.html")
m_all.save(city_html)
print(f"Saved city-wise score map to: {city_html}")

m_all

Saved city-wise score map to: c:\Users\harry\Desktop\sig_datathon\Connectivity\connectivity_citywise_score_map.html


In [ ]:
# Single-grid drill-down compatible with current result dataframe
def drill_down_grid(
    result_df,
    stations_gdf,
    grid_index=None,
    lon=None,
    lat=None,
    zoom_start=13,
    save_html=True,
    result_projected_crs="EPSG:27700",
):
    required_cols = ["connectivity_score"]
    missing = [c for c in required_cols if c not in result_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Detect all nearest-station columns so this works for any k.
    station_cols = []
    for col in result_df.columns:
        if col.startswith("nearest_station_") and col[len("nearest_station_"):].isdigit():
            station_cols.append(col)

    station_cols = sorted(station_cols, key=lambda c: int(c.split("_")[-1]))
    if not station_cols:
        raise ValueError("No nearest_station_<n> columns found in result_df.")

    # Build a map-ready version of result_df in EPSG:4326.
    if isinstance(result_df, gpd.GeoDataFrame) and "geometry" in result_df.columns and result_df.crs is not None:
        result_map = result_df.to_crs(epsg=4326).copy()
    elif {"x", "y"}.issubset(result_df.columns):
        result_map = gpd.GeoDataFrame(
            result_df.copy(),
            geometry=gpd.points_from_xy(result_df["x"], result_df["y"]),
            crs=result_projected_crs,
        ).to_crs(epsg=4326)
    elif {"lon", "lat"}.issubset(result_df.columns):
        is_projected_like = (result_df["lon"].abs().max() > 180) or (result_df["lat"].abs().max() > 90)
        in_crs = result_projected_crs if is_projected_like else "EPSG:4326"
        result_map = gpd.GeoDataFrame(
            result_df.copy(),
            geometry=gpd.points_from_xy(result_df["lon"], result_df["lat"]),
            crs=in_crs,
        ).to_crs(epsg=4326)
    else:
        raise ValueError("result_df must contain geometry, x/y, or lon/lat columns")

    if grid_index is None:
        if lon is None or lat is None:
            raise ValueError("Provide grid_index, or both lon and lat (EPSG:4326).")
        d2 = (result_map.geometry.x - float(lon)) ** 2 + (result_map.geometry.y - float(lat)) ** 2
        grid_index = int(d2.idxmin())

    row = result_map.loc[grid_index]
    center = [float(row.geometry.y), float(row.geometry.x)]
    print(f"Map center (lat, lon): {center[0]:.6f}, {center[1]:.6f}")

    # Ensure stations are map-ready for folium (lat/lon)
    if getattr(stations_gdf, "crs", None) is not None and stations_gdf.crs.to_epsg() != 4326:
        stations_map = stations_gdf.to_crs(epsg=4326)
    else:
        stations_map = stations_gdf

    # Use explicit tile layers for more reliable rendering in notebook and exported HTML
    m = folium.Map(location=center, zoom_start=zoom_start, tiles=None)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(m)
    folium.TileLayer("CartoDB positron", name="CartoDB Positron", control=True).add_to(m)

    grid_popup = (
        f"grid_index={grid_index}<br>"
        f"score={row['connectivity_score']:.4f}<br>"
        f"line_unique_count={row.get('line_unique_count', np.nan)}<br>"
        f"harmonic_mean_adj_dist={row.get('harmonic_mean_adj_dist', np.nan):.4f}"
    )

    folium.CircleMarker(
        location=center,
        radius=8,
        color="red",
        fill=True,
        fill_opacity=0.95,
        popup=folium.Popup(grid_popup, max_width=300),
    ).add_to(m)

    bounds = [center]
    for station_col in station_cols:
        rank = int(station_col.split("_")[-1])
        station_name = row.get(station_col)
        if pd.isna(station_name):
            continue

        station_match = stations_map[stations_map["NAME"] == station_name]
        if station_match.empty:
            continue

        station_pt = station_match.iloc[0].geometry
        s_lat = float(station_pt.y)
        s_lon = float(station_pt.x)
        bounds.append([s_lat, s_lon])

        dist_col = f"nearest_station_{rank}_dist"
        dist_val = row.get(dist_col, np.nan)

        folium.Marker(
            location=[s_lat, s_lon],
            popup=f"{rank}. {station_name}<br>adj_dist={dist_val:.2f}",
            icon=folium.Icon(color="blue", icon="info-sign"),
        ).add_to(m)

        folium.PolyLine(
            locations=[center, [s_lat, s_lon]],
            color="gray",
            weight=2,
            opacity=0.8,
        ).add_to(m)

    if len(bounds) > 1:
        m.fit_bounds(bounds, padding=(25, 25))

    folium.LayerControl(collapsed=False).add_to(m)

    if save_html:
        single_html = os.path.abspath(f"connectivity_single_grid_{grid_index}.html")
        m.save(single_html)
        print(f"Saved single-grid map to: {single_html}")

    return m

# Example: drill into the highest-scoring grid
top_idx = int(result["connectivity_score"].idxmin())
m_single = drill_down_grid(result, gdf, grid_index=top_idx, zoom_start=13, save_html=True)
m_single

Map center (lat, lon): 51.519481, -0.138477
Saved single-grid map to: c:\Users\harry\Desktop\sig_datathon\Connectivity\connectivity_single_grid_174.html
